<a href="https://colab.research.google.com/github/dehigher/ai-practice/blob/main/3_%E5%88%9D%E8%AF%86LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip3 install langchain

### 提示词模版

In [3]:
!pip3 install langchain-deepseek

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.2 MB/s eta 0:00:00


In [ ]:
import os
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model

os.environ["DEEPSEEK_API_KEY"] = "sk-3547bbf4b0da4f8eb5a99a049e2614d4"

model = init_chat_model(
    model="deepseek-chat",
    api_base="https://api.deepseek.com"
)

# 构造模版提示词
prompt_template = "What is a good name for a company that makes {product}? And only return the best one."
prompt = PromptTemplate.from_template(prompt_template)

# 输入参数，单个调用
singleInput = prompt.format(product='colorful socks')
resp = model.invoke(singleInput)
print("single:{}".format(resp.content))


# 输入参数，批量调用
products = [
    "cloudnative devops platform",
    "Noise cancellation headphone",
    "colorful socks",
]
for p in products:
  batchInput = prompt.format(product=p)
  resp = model.invoke(batchInput)
  print("batchInput:{}".format(resp.content))

single:ChromaSox
batchInput:**Nexus**
batchInput:Aether
batchInput:ChromaToes


## HTTP request chain

In [ ]:
!pip install langchain-deepseek

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.0 MB/s eta 0:00:00


In [ ]:
import os
import requests
from textwrap import dedent
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model

os.environ["DEEPSEEK_API_KEY"] = "sk-3547bbf4b0da4f8eb5a99a049e2614d4"



model = init_chat_model(
    model="deepseek-chat",
    api_base="https://api.deepseek.com"
)

# 构造模版提示词，dedent格式化字符串，对齐作用
template = dedent(
    """
      Between >>> and <<< are the raw search result text from web.
      Extract the answer to the question '{query}' or say "not found" if the information is not contained.
      Use the format
      Extracted:<answer or "not found">
      >>> {requests_result} <<<
      Extracted:
    """
)


prompt = PromptTemplate(
    input_variables = ["query", "requests_result" ],
    template = template
)

def query_build(question):
  question = question.replace(" ", "+")
  url = "http://www.baidu.com/s?wd=" + question
  resp = requests.get(url=url, timeout=10)
  resp.raise_for_status() # 检查响应状态码抛异常
  search_text = resp.text[:8000] #截断一下避免太长

  result = model.invoke(prompt.format(
      query = question,
      requests_result = search_text
  ))

  # result 可能是 BaseMessage / str，做一下兼容
  if hasattr(result, 'content'):
    return result.content
  return result


print(query_build("今天天气怎样"))

Extracted: not found


## 基于langchain实现调用链

### 1、LCEL： langchain expression language
核心思想是：把“提示词、模型、解析器、以及一些数据变换”都当成 Runnable（可运行组件），然后用很直观的运算符把它们“拼管道”。
如： chain = prompt | llm | StrOutputParser()
1. prompt: 把输入字典（如 {"question": "..."}）渲染成一组 Chat 消息（system/user）
2. llm: 把消息发给模型，得到模型输出（一般是 AIMessage）
3. StrOutputParser()：把 AIMessage 解析成纯字符串 str

### 2、RunnablePassthrough.assign
pipeline = RunnablePassthrough.assign(answer_zh=chain1) | chain2
1. RunnablePassthrough
意思是“先原样把输入传下去”（不改变输入）。
假设原始输入是 {"question": "用一句话解释什么是链式调用？"}
2. assign(answer_zh=chain1)
意思是：在保留原输入的同时，新增一个 key：answer_zh，它的值由 chain1 算出来
3. | chian2
然后把这个“带有 answer_zh 的字典”交给 chain2
而 chain2 的 prompt 里写了 {answer_zh}，所以它就能拿到中文回答并翻译



In [6]:
"""
LangChain 链式调用 Demo（Python + deepseek-chat）

需求：
1) chain1：根据问题写简要中文回答
2) chain2：把 chain1 的中文回答翻译成英文
3) 使用 DeepSeek 的 deepseek-chat 模型（OpenAI-compatible 接口）
4) 使用 LangChain LCEL（Runnable）把两个 chain 串起来

环境变量：
- DEEPSEEK_API_KEY: 你的 DeepSeek API Key
- DEEPSEEK_BASE_URL: DeepSeek OpenAI-compatible base_url（默认 https://api.deepseek.com/v1）
"""

import os
from dotenv import load_dotenv

# DeepSeek 为 OpenAI-compatible 接口时，通常用 langchain_openai.ChatOpenAI 进行对接
from langchain_openai import ChatOpenAI

# Prompt 相关：使用 ChatPromptTemplate 构造“system + human”消息
from langchain_core.prompts import ChatPromptTemplate

# 输出解析：把模型输出（AIMessage）解析成纯字符串
from langchain_core.output_parsers import StrOutputParser

# Runnable 相关：用于在链中透传输入、拼接中间结果等（LCEL 写法）
from langchain_core.runnables import RunnablePassthrough

# -> ChatOpenAI 是返回值注解，也可以省略
def build_llm() -> ChatOpenAI:
    return ChatOpenAI(
        model="deepseek-chat",
        api_key="sk-3547bbf4b0da4f8eb5a99a049e2614d4",
        base_url="https://api.deepseek.com",
        temperature=0.2, # temperature 越低回答越稳定；这里选择 0.2
    )


def build_chain1(llm: ChatOpenAI):
    """
      chain1：输入 question -> 输出简要中文回答字符串
      LCEL 组合方式：
      prompt | llm | output_parser
    """
    # system：约束风格；human：接收用户变量 {question}
    prompt_answer_zh = ChatPromptTemplate.from_messages(
        [
            ("system", "你是一个严谨的助手，请用中文给出简要回答，限制在80字以内。"),
            ("human", "问题：{question}"),
        ]
    )
    # StrOutputParser 将模型输出统一变成 str
    return prompt_answer_zh | llm | StrOutputParser()


def build_chain2(llm: ChatOpenAI):
    """
    chain2：输入 answer_zh（中文回答）-> 输出英文翻译字符串
    """
    prompt_translate_en = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a professional translator. Translate the given Chinese into concise English."),
            ("human", "Chinese: {answer_zh}"),
        ]
    )

    return prompt_translate_en | llm | StrOutputParser()


# 将 chain1 和 chain2 串起来。
def build_chained_pipeline(chain1, chain2):
    return RunnablePassthrough.assign(answer_zh=chain1) | chain2


def main():
    """
    主函数：加载环境变量 -> 构建 LLM -> 构建 chain1/chain2 -> 串联 -> invoke 测试
    """

    llm = build_llm()
    chain1 = build_chain1(llm)
    chain2 = build_chain2(llm)

    chained = build_chained_pipeline(chain1, chain2)

    # 示例问题（你也可以替换成任意问题）
    question = "用一句话解释什么是链式调用？"

    # invoke 输入必须与 prompt 模板变量匹配：这里需要 question
    result_en = chained.invoke({"question": question})

    print("Question:", question)
    print("English:", result_en)


if __name__ == "__main__":
    main()

Question: 用一句话解释什么是链式调用？
English: Method chaining is a programming technique that allows multiple methods to be called sequentially on a single object, with each method returning the object itself to enable further calls.
